In [1]:
from datatools import *
from factortools import MAD_winsorize
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
def __reg(df):
    y = df['sub_MIDCAP'].values
    X = np.c_[np.ones((len(df), 1)), df['LNCAP'].values]
    W = np.diag(np.sqrt(df['circ_mv']))
    b = np.linalg.pinv(X.T@W@X)@X.T@W@y
# 去除极端值
    resi = MAD_winsorize(y - X@b, multiplier=5)
# 标准化
    resi -= np.nanmean(resi)
    resi /= np.nanstd(resi)
    return pd.Series(resi, index=df['ts_code'])

In [ ]:
df = get_basic(start_date='2014-01-01', end_date='2025-12-31',fields=['circ_mv'])
df['circ_mv'] = df['circ_mv']/1e4
df['LNCAP'] = np.log(df['circ_mv'])
df['sub_MIDCAP'] = df['LNCAP']**3
# 截面回归正交化处理
MIDCAP = df.groupby('trade_date').apply(__reg)
MIDCAP.name = 'MIDCAP'
df = df.merge(MIDCAP.reset_index())

In [ ]:
price = get_price(start_date='2014-01-01', end_date='2025-12-31', fields=['close','pre_close'])
hs300 = get_index_K(codes=['000300.SH'], start_date='2014-01-01', end_date='2025-12-31', fields=['close','pre_close'])
hs300['trade_date'] = pd.to_datetime(hs300['trade_date'])
price = pd.concat([price, hs300]).reset_index(drop=True)
close = pd.pivot_table(price, values='close', index='trade_date', columns='ts_code').ffill()
pre_close = pd.pivot_table(price, values='pre_close', index='trade_date', columns='ts_code').bfill()
ret = close / pre_close - 1

In [53]:
window=252
half_life=63
W = 0.5 ** (np.arange(window)[::-1] / half_life)

In [ ]:
# 计算BETA和Hist_sigma
beta, hist_sigma = [], []
for i in tqdm(range(len(ret)-window+1), description='正在计算beta...'):
    tmp = ret.iloc[i:i+window, :].copy()
    W_full = np.diag(W)
    Y_full = tmp.dropna(axis=1).drop(columns='000300.SH')
    idx_full, Y_full = Y_full.columns, Y_full.values
    X_full = np.c_[np.ones((window, 1)), tmp.loc[:, '000300.SH'].values]
    beta_full = np.linalg.pinv(X_full.T@W_full@X_full)@X_full.T@W_full@Y_full
    hist_sigma_full = pd.Series(np.std(Y_full - X_full@beta_full, axis=0), index=idx_full, name=tmp.index[-1])
    beta_full = pd.Series(beta_full[1], index=idx_full, name=tmp.index[-1])
    
    beta_lack, hist_sigma_lack = {}, {}
    for c in set(tmp.columns) - set(idx_full) - set('000300.SH'):
        tmp_ = tmp.loc[:, [c, '000300.SH']].copy()
        tmp_.loc[:, 'W'] = W
        tmp_ = tmp_.dropna()
        if len(tmp_) < half_life:
            continue
        W_lack = np.diag(tmp_['W'])
        X_lack = np.c_[np.ones(len(tmp_)), tmp_['000300.SH'].values]
        Y_lack = tmp_[c].values
        beta_tmp = np.linalg.pinv(X_lack.T@W_lack@X_lack)@X_lack.T@W_lack@Y_lack
        hist_sigma_lack[c] = np.std(Y_lack - X_lack@beta_tmp)
        beta_lack[c] = beta_tmp[1]
    beta_lack = pd.Series(beta_lack, name=tmp.index[-1])
    hist_sigma_lack = pd.Series(hist_sigma_lack, name=tmp.index[-1])
    beta.append(pd.concat([beta_full, beta_lack]).sort_index())
    hist_sigma.append(pd.concat([hist_sigma_full, hist_sigma_lack]).sort_index())
beta = pd.concat(beta, axis=1).T
beta = pd.melt(beta.reset_index(), id_vars='index').dropna()
beta.columns = ['trade_date', 'ts_code', 'BETA']
hist_sigma = pd.concat(hist_sigma, axis=1).T
hist_sigma = pd.melt(hist_sigma.reset_index(), id_vars='index').dropna()
hist_sigma.columns = ['trade_date', 'ts_code', 'Hist_sigma']
factor = pd.merge(beta, hist_sigma)

In [121]:
daily_std = ret.ewm(halflife=42, min_periods=252,adjust=False).std()
daily_std.index.name = 'trade_date'
daily_std = pd.melt(daily_std.reset_index(), id_vars='trade_date', value_name='Daily_std').dropna()

In [ ]:
idx = close.index
CMRA = {}
for i in tqdm(range(len(close)-window+1), desc='正在计算CMRA...'):
    close_ = close.iloc[i:i+window, :]
    pre_close_ = pre_close.iloc[i, :]
    pre_close_.name = pre_close_.name - pd.Timedelta(days=1)
    close_ = close_.append(pre_close_).sort_index().iloc[list(range(0, 253, 21)), :]
    r_tau = close_.pct_change().dropna(how='all')
    Z_T = np.log(r_tau+1).cumsum(axis=0)
    CMRA[idx[i+window-1]] = Z_T.max(axis=0) - Z_T.min(axis=0)

In [ ]:
CMRA = pd.DataFrame(CMRA).T
CMRA.index.name = 'trade_date'
CMRA = pd.melt(CMRA.reset_index(), id_vars='trade_date', value_name='Cumulative_range').dropna()